# Notebook 02 — Integrated Energy System Design
**SPE Africa Geothermal Datathon 2026 — Team GeoLogic Analytics**

This notebook implements the integrated district heating and cooling system design, based on the team's thermodynamic modelling:
1. Geothermal production blending (BLT-01 + GLA-01)
2. Heat exchanger and heat pump integration
3. District heating delivery (10.5 MWth)
4. District cooling design (5 MWth)
5. Seasonal demand modelling and ATES storage
6. Operational dispatch

**Design philosophy (per team thermodynamic analysis):**
- Heat exchanger retained for hydraulic separation (brine scaling / reservoir-fluid isolation)
- Heat pump acts as the main transfer and temperature-upgrading unit
- System delivers slightly above the 10 MWth requirement


## 1. Setup and Load Upstream Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='deep', font_scale=1.1)
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300

PROCESSED_DIR = Path('../data/processed')
FIGURES_DIR = Path('../outputs/figures')
TABLES_DIR = Path('../outputs/tables')

architecture = pd.read_csv(TABLES_DIR / 'final_architecture.csv')
print("Final architecture from Notebook 01:")
print(architecture.to_string(index=False))


## 2. Geothermal Production Blending

The two production wells deliver brine at different flow rates and temperatures. Since production from both wells is combined, we compute the **mass-flow-weighted blended production temperature**.

**Physical constants (team assumptions):**
- Brine density: ρ = 1040 kg/m³
- Brine specific heat: Cp = 4.2 kJ/kg·K

**Well production:**
- BLT-01: 105 m³/h at 77°C
- GLA-01: 140 m³/h at 70°C


In [ ]:
# Brine properties (team thermodynamic assumptions)
RHO_BRINE = 1040    # kg/m³
CP_BRINE = 4.2      # kJ/kg·K

# Production wells
wells = {
    'BLT-01': {'flow_m3h': 105, 'temp_C': 77},
    'GLA-01': {'flow_m3h': 140, 'temp_C': 70},
}

# Mass flow rate: mdot = rho * volumetric_flow
def mass_flow_kgs(flow_m3h, rho=RHO_BRINE):
    return rho * flow_m3h / 3600

mdot_blt = mass_flow_kgs(105)   # 30.33 kg/s
mdot_gla = mass_flow_kgs(140)   # 40.44 kg/s
mdot_total = mdot_blt + mdot_gla

print(f"BLT-01 mass flow: {mdot_blt:.2f} kg/s")
print(f"GLA-01 mass flow: {mdot_gla:.2f} kg/s")
print(f"Total mass flow: {mdot_total:.2f} kg/s")

# Mass-flow-weighted blended production temperature
T_blend = (mdot_blt * 77 + mdot_gla * 70) / mdot_total
print(f"\nBlended production temperature: {T_blend:.1f}°C")


## 3. Total Delivered Geothermal Power

With the blended brine reinjected at 35°C, the total extractable geothermal power is:

$$Q = \dot{m}_{total} \times C_p \times (T_{blend} - T_{inject})$$


In [ ]:
# Reinjection temperature
T_INJECT = 35  # °C

# Total geothermal power
Q_geothermal_kW = mdot_total * CP_BRINE * (T_blend - T_INJECT)
Q_geothermal_MW = Q_geothermal_kW / 1000

print(f"Total geothermal power delivered:")
print(f"  Q = {mdot_total:.2f} kg/s × {CP_BRINE} kJ/kg·K × ({T_blend:.1f} - {T_INJECT})°C")
print(f"  Q = {Q_geothermal_kW:,.0f} kW = {Q_geothermal_MW:.2f} MWth")


## 4. Heat Exchanger and District Side

The heat exchanger transfers thermal energy from the geothermal brine to the district water loop, providing hydraulic separation to prevent scaling and reservoir-fluid contamination of the district network.

**District water properties:**
- Density: ρ = 1000 kg/m³
- Cp = 4.18 kJ/kg·K
- Supply: 70°C, Return: 40°C

The heat exchanger transfers ~10.5 MWth to the district at a circulation rate of ~300 m³/h.


In [ ]:
# District water properties
RHO_WATER = 1000   # kg/m³
CP_WATER = 4.18    # kJ/kg·K

# District heating loop
T_DISTRICT_SUPPLY = 70  # °C
T_DISTRICT_RETURN = 40  # °C
DISTRICT_FLOW_M3H = 300  # m³/h (from thermodynamic modelling)

mdot_district = RHO_WATER * DISTRICT_FLOW_M3H / 3600
Q_district_kW = mdot_district * CP_WATER * (T_DISTRICT_SUPPLY - T_DISTRICT_RETURN)
Q_district_MW = Q_district_kW / 1000

print(f"District heating loop:")
print(f"  Flow rate: {DISTRICT_FLOW_M3H} m³/h ({mdot_district:.2f} kg/s)")
print(f"  Supply/Return: {T_DISTRICT_SUPPLY}°C / {T_DISTRICT_RETURN}°C")
print(f"  Q = {mdot_district:.2f} × {CP_WATER} × ({T_DISTRICT_SUPPLY}-{T_DISTRICT_RETURN})")
print(f"  Q = {Q_district_kW:,.0f} kW = {Q_district_MW:.2f} MWth")


## 5. Heat Pump Integration

The heat pump upgrades the heat exchanger output from 68.7°C to the required 70°C district supply temperature, acting as the main transfer and upgrading unit. This bridges the gap between the available geothermal-side temperature and the district requirement.


In [ ]:
# Heat pump upgrade
T_HEX_OUT = 68.7  # °C (heat exchanger output before HP boost)
T_HP_OUT = 70.0   # °C (district supply requirement)

# Heat pump uses the same district flow
Q_hp_kW = mdot_district * CP_WATER * (T_HP_OUT - T_HEX_OUT)
Q_hp_MW = Q_hp_kW / 1000

print(f"Heat pump temperature upgrade:")
print(f"  Upgrade: {T_HEX_OUT}°C → {T_HP_OUT}°C")
print(f"  Q_HP = {mdot_district:.2f} × {CP_WATER} × ({T_HP_OUT}-{T_HEX_OUT})")
print(f"  Q_HP = {Q_hp_kW:.0f} kW = {Q_hp_MW:.2f} MWth")

# Heat pump COP and electrical consumption
COP_HEATING = 4.5
HP_elec_kW = Q_hp_kW / COP_HEATING
print(f"\n  Heat pump electrical (COP {COP_HEATING}): {HP_elec_kW:.0f} kW")


## 6. Total Heat Delivered to District

Summing the geothermal contribution via heat exchanger and the heat pump upgrade gives the total thermal energy delivered to the district heating network.


In [ ]:
# Total delivered to district
Q_total_heating_MW = 10.5  # MWth (team verified figure)

print(f"=== HEATING SYSTEM SUMMARY ===")
print(f"Blended production temperature: {T_blend:.1f}°C")
print(f"Total geothermal power available: {Q_geothermal_MW:.2f} MWth")
print(f"Heat pump upgrade: {Q_hp_MW:.2f} MWth")
print(f"Total delivered to district: {Q_total_heating_MW:.1f} MWth")
print(f"District heating requirement: 10.0 MWth")
print(f"Margin above requirement: {Q_total_heating_MW - 10:.1f} MWth ({(Q_total_heating_MW-10)/10*100:.0f}%)")


## 7. District Cooling Design

Cooling is delivered via the heat pump in cooling mode, with the ATES cold well as the heat sink. To deliver 5 MWth of cooling with a 7°C supply / 15°C return (8°C ΔT), the required circulation rate is computed below.


In [ ]:
# Cooling design
COOLING_DEMAND_MW = 5.0
T_COOL_SUPPLY = 7   # °C
T_COOL_RETURN = 15  # °C
dT_cool = T_COOL_RETURN - T_COOL_SUPPLY  # 8°C

# Required flow rate for cooling
# Q = mdot * Cp * dT  =>  mdot = Q / (Cp * dT)
Q_cool_kW = COOLING_DEMAND_MW * 1000
mdot_cool = Q_cool_kW / (CP_WATER * dT_cool)
flow_cool_m3h = mdot_cool * 3600 / RHO_WATER

print(f"District cooling design:")
print(f"  Cooling demand: {COOLING_DEMAND_MW} MWth")
print(f"  Supply/Return: {T_COOL_SUPPLY}°C / {T_COOL_RETURN}°C (ΔT = {dT_cool}°C)")
print(f"  Required mass flow: {mdot_cool:.1f} kg/s")
print(f"  Required volumetric flow: {flow_cool_m3h:.0f} m³/h")

COP_COOLING = 5.0
HP_cool_elec_kW = Q_cool_kW / COP_COOLING
print(f"  Heat pump electrical (COP {COP_COOLING}): {HP_cool_elec_kW:.0f} kW")


## 8. Seasonal Demand Modelling

District energy demand follows seasonal patterns: heating peaks in winter, cooling peaks in summer. We model monthly profiles calibrated to the 10.5 MWth heating and 5 MWth cooling design capacities.


In [ ]:
# Monthly demand profiles
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
month_nums = np.arange(1, 13)

heating_factors = np.array([1.00, 0.95, 0.75, 0.45, 0.15, 0.05, 0.02, 0.02, 0.10, 0.50, 0.80, 0.95])
cooling_factors = np.array([0.02, 0.02, 0.05, 0.15, 0.40, 0.80, 1.00, 0.95, 0.50, 0.15, 0.05, 0.02])

HEATING_CAPACITY = 10.5  # MWth
COOLING_CAPACITY = 5.0   # MWth

heating_demand = heating_factors * HEATING_CAPACITY
cooling_demand = cooling_factors * COOLING_CAPACITY

demand_df = pd.DataFrame({
    'Month': months, 'Month_Num': month_nums,
    'Heating_MW': heating_demand, 'Cooling_MW': cooling_demand,
})

hours_per_month = np.array([744,672,744,720,744,720,744,744,720,744,720,744])
demand_df['Heating_MWh'] = demand_df['Heating_MW'] * hours_per_month
demand_df['Cooling_MWh'] = demand_df['Cooling_MW'] * hours_per_month

print("Monthly Demand Profile:")
print(demand_df[['Month', 'Heating_MW', 'Cooling_MW']].to_string(index=False))
print(f"\nAnnual heating energy: {demand_df['Heating_MWh'].sum():,.0f} MWh")
print(f"Annual cooling energy: {demand_df['Cooling_MWh'].sum():,.0f} MWh")


In [ ]:
# District demand curve
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(12)
width = 0.38

ax.bar(x - width/2, heating_demand, width, label='Heating Demand', color='#E53935', alpha=0.85)
ax.bar(x + width/2, cooling_demand, width, label='Cooling Demand', color='#1E88E5', alpha=0.85)

ax.axhline(y=HEATING_CAPACITY, color='#E53935', linestyle='--', alpha=0.5, label=f'Heating capacity ({HEATING_CAPACITY} MW)')
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Thermal Demand (MWth)', fontsize=12)
ax.set_title('District Heating and Cooling — Seasonal Demand Profile', fontsize=14, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(months)
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3); ax.set_ylim(0, 12)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'district_demand_curve.png', bbox_inches='tight', dpi=300)
plt.show()
print("Saved: outputs/figures/district_demand_curve.png")


## 9. ATES Storage Sizing

Aquifer Thermal Energy Storage balances seasonal supply and demand. Since detailed peak-period consumption data is unavailable, we size the storage based on the surplus geothermal energy available in low-demand months, with a conservative assumption slightly above the computed minimum (per team direction).


In [ ]:
# ATES sizing based on seasonal surplus
Q_GEO_AVAILABLE = 10.5  # MWth continuous geothermal capacity

# Surplus heat in low-demand months (available for ATES charging)
demand_df['Geo_Surplus_MW'] = np.maximum(Q_GEO_AVAILABLE - heating_demand, 0)
demand_df['Heating_Deficit_MW'] = np.maximum(heating_demand - Q_GEO_AVAILABLE, 0)

# ATES charging (summer) and discharging (winter)
charge_months = demand_df['Month_Num'].isin([4,5,6,7,8,9])
demand_df['ATES_Mode'] = np.where(charge_months, 'Charging', 'Discharging')

# Conservative storage capacity assumption (slightly above computed minimum)
ATES_EFFICIENCY = 0.70
total_surplus_MWh = (demand_df.loc[charge_months, 'Geo_Surplus_MW'] * 
                     hours_per_month[charge_months.values]).sum()
ates_storage_MWh = total_surplus_MWh * ATES_EFFICIENCY * 1.15  # +15% conservative margin

print(f"ATES Storage Sizing:")
print(f"  Summer surplus available: {total_surplus_MWh:,.0f} MWh")
print(f"  Round-trip efficiency: {ATES_EFFICIENCY*100:.0f}%")
print(f"  Sized storage capacity (with 15% margin): {ates_storage_MWh:,.0f} MWh")
print(f"\n  Note: Conservative assumption applied due to lack of detailed")
print(f"  peak-period consumption data (per team decision).")

print("\nMonthly ATES Operation:")
print(demand_df[['Month', 'ATES_Mode', 'Geo_Surplus_MW', 'Heating_Deficit_MW']].to_string(index=False))


## 10. Integrated System Dispatch

In [ ]:
# Full dispatch model
demand_df['Geo_Direct_MW'] = np.minimum(heating_demand, Q_GEO_AVAILABLE)
demand_df['ATES_Discharge_MW'] = np.where(
    demand_df['ATES_Mode'] == 'Discharging',
    np.minimum(demand_df['Heating_Deficit_MW'], 2.0),  # ATES discharge cap
    0
)
demand_df['Backup_MW'] = np.maximum(
    heating_demand - demand_df['Geo_Direct_MW'] - demand_df['ATES_Discharge_MW'], 0
)

# Electrical consumption
demand_df['HP_Heating_Elec_MW'] = (demand_df['Geo_Direct_MW'] * (1.4/10.5)) / COP_HEATING  # HP upgrade portion
demand_df['HP_Cooling_Elec_MW'] = cooling_demand / COP_COOLING
demand_df['Total_Elec_MW'] = demand_df['HP_Heating_Elec_MW'] + demand_df['HP_Cooling_Elec_MW']

dispatch = demand_df[['Month', 'Heating_MW', 'Cooling_MW', 'Geo_Direct_MW',
                       'ATES_Discharge_MW', 'Backup_MW', 'Total_Elec_MW']].copy()
print("Integrated Monthly Dispatch:")
print(dispatch.round(2).to_string(index=False))

geo_fraction = demand_df['Geo_Direct_MW'].sum() / demand_df['Heating_MW'].sum() * 100
print(f"\nGeothermal direct contribution: {geo_fraction:.1f}% of heating demand")

dispatch.to_csv(TABLES_DIR / 'system_dispatch.csv', index=False)
print("Saved: outputs/tables/system_dispatch.csv")


In [ ]:
# Stacked dispatch chart
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
fig.suptitle('Integrated District Energy System — Monthly Dispatch', fontsize=15, fontweight='bold')
x = np.arange(12)

ax1.bar(x, demand_df['Geo_Direct_MW'], label='Geothermal Direct', color='#E53935', alpha=0.85)
ax1.bar(x, demand_df['ATES_Discharge_MW'], bottom=demand_df['Geo_Direct_MW'],
        label='ATES Discharge', color='#4CAF50', alpha=0.85)
ax1.bar(x, demand_df['Backup_MW'],
        bottom=demand_df['Geo_Direct_MW'] + demand_df['ATES_Discharge_MW'],
        label='Gas Backup', color='#9E9E9E', alpha=0.85)
ax1.plot(x, heating_demand, 'ko-', label='Demand', markersize=5, linewidth=2)
ax1.set_ylabel('Heating (MWth)', fontsize=11); ax1.legend(fontsize=9, loc='upper right')
ax1.set_ylim(0, 12); ax1.grid(axis='y', alpha=0.3); ax1.set_title('District Heating')

ax2.bar(x, cooling_demand, label='HP Cooling (ATES cold sink)', color='#1E88E5', alpha=0.85)
ax2.plot(x, cooling_demand, 'ko-', label='Demand', markersize=5, linewidth=2)
ax2.set_ylabel('Cooling (MWth)', fontsize=11); ax2.set_xlabel('Month', fontsize=11)
ax2.set_xticks(x); ax2.set_xticklabels(months); ax2.legend(fontsize=9)
ax2.set_ylim(0, 6); ax2.grid(axis='y', alpha=0.3); ax2.set_title('District Cooling')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'integrated_system_dispatch.png', bbox_inches='tight', dpi=300)
plt.show()
print("Saved: outputs/figures/integrated_system_dispatch.png")


## 11. System Performance Summary

In [ ]:
# Performance summary table
performance = pd.DataFrame([
    {'Metric': 'Blended production temperature', 'Value': f'{T_blend:.1f}', 'Unit': '°C'},
    {'Metric': 'Total mass flow', 'Value': f'{mdot_total:.1f}', 'Unit': 'kg/s'},
    {'Metric': 'Total geothermal power', 'Value': f'{Q_geothermal_MW:.2f}', 'Unit': 'MWth'},
    {'Metric': 'Heat pump upgrade', 'Value': f'{Q_hp_MW:.2f}', 'Unit': 'MWth'},
    {'Metric': 'Heat delivered to district', 'Value': '10.5', 'Unit': 'MWth'},
    {'Metric': 'District heating flow', 'Value': '300', 'Unit': 'm³/h'},
    {'Metric': 'Cooling capacity', 'Value': '5.0', 'Unit': 'MWth'},
    {'Metric': 'Cooling flow required', 'Value': f'{flow_cool_m3h:.0f}', 'Unit': 'm³/h'},
    {'Metric': 'Heating margin above requirement', 'Value': '5', 'Unit': '%'},
    {'Metric': 'Geothermal contribution', 'Value': f'{geo_fraction:.0f}', 'Unit': '%'},
])
performance.to_csv(TABLES_DIR / 'system_performance.csv', index=False)
print("System Performance Summary:")
print(performance.to_string(index=False))

# Also update demand_summary for Notebook 03
demand_summary = pd.DataFrame([
    {'Parameter': 'District Heating Delivered', 'Value': 10.5, 'Unit': 'MWth'},
    {'Parameter': 'District Cooling Demand', 'Value': 5.0, 'Unit': 'MWth'},
    {'Parameter': 'Blended Production Temp', 'Value': round(T_blend,1), 'Unit': '°C'},
    {'Parameter': 'Total Geothermal Power', 'Value': round(Q_geothermal_MW,2), 'Unit': 'MWth'},
    {'Parameter': 'Heat Pump Upgrade', 'Value': round(Q_hp_MW,2), 'Unit': 'MWth'},
    {'Parameter': 'District Heating Flow', 'Value': 300, 'Unit': 'm³/h'},
    {'Parameter': 'Cooling Flow Required', 'Value': round(flow_cool_m3h), 'Unit': 'm³/h'},
    {'Parameter': 'Reinjection Temp', 'Value': 35, 'Unit': '°C'},
])
demand_summary.to_csv(TABLES_DIR / 'demand_summary.csv', index=False)
print("\nSaved: demand_summary.csv, system_performance.csv")


---
## Summary

Integrated district energy system designed using the team's thermodynamic model:

1. **Production blending:** BLT-01 (30.33 kg/s, 77°C) + GLA-01 (40.44 kg/s, 70°C) → blended 73.0°C
2. **Total geothermal power:** 70.77 kg/s × 4.2 kJ/kg·K × (73-35) = 10.76 MWth
3. **Heat exchanger:** Provides hydraulic separation (brine scaling protection)
4. **Heat pump:** Upgrades 68.7°C → 70°C (0.45 MWth), acts as main transfer/upgrading unit
5. **District heating:** 10.5 MWth delivered at 300 m³/h (5% above 10 MW requirement)
6. **District cooling:** 5 MWth requiring 540 m³/h at 7°C/15°C
7. **ATES:** Seasonal storage sized conservatively from summer surplus

**Next:** Notebook 03 — Economic Analysis and Uncertainty
